Task 1. 
DNA / Protein Sequence analysis
- Download from NCBI or Uniport
- Perform BLAST analysis to find homologous sequence + Document results (similarity, identity, alignment score) 2-3 pages

1. Installation
2. Download sequences from Uniprot and NCBI
3. Working with FASTQ and FASTA files 
4. Running BLAST
5. Parsing results --> Similarity, identity and scores
6. Full report 

In [2]:
# installation 
%pip install -q biopython requests pandas matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Importing 
import os
import time
from datetime import datetime

import requests
import pandas as pd
import matplotlib.pyplot as plt

from Bio import Entrez, SeqIO
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# configuration
Entrez.email = "tamiminezhad.m7@gmail.com"

_api_key = os.environ.get("NCBI_API_KEY")
if _api_key:
    Entrez.api_key = _api_key

OUTPUT_DIR = "blast_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.set_option("display.max_colwidth", 75)

In [4]:
# Downloading NCBI

def download_from_ncbi(accession, db="protein", rettype="fasta",
                       retmode="text", filename=None):

    if filename is None:
        ext = "fasta" if rettype == "fasta" else "gb"
        filename = os.path.join(OUTPUT_DIR, accession.replace(".", "_") + "." + ext)

    print("Fetching %s from NCBI db='%s' ..." % (accession, db))

    with Entrez.efetch(db=db, id=accession, rettype=rettype, retmode=retmode) as handle:
        data = handle.read()

    if isinstance(data, bytes):                      
        data = data.decode("utf-8")

    if data.lstrip().startswith("<"):                
        raise RuntimeError("NCBI returned an error page — check the accession "
                           "and your e-mail/API key configuration.")

    with open(filename, "w") as f:
        f.write(data)

    print("Saved %s characters to %s" % (format(len(data), ","), filename))
    return filename


In [5]:
# download test protein with accession
ncbi_protein_file = download_from_ncbi(
    accession="NP_000537.3",
    db="protein",
    rettype="fasta",
    filename=os.path.join(OUTPUT_DIR, "TP53_NCBI_protein.fasta"),
)

# download test Gene
ncbi_gene_file = download_from_ncbi(
    accession="NM_007294.4",
    db="nucleotide",
    rettype="fasta",
    filename=os.path.join(OUTPUT_DIR, "BRCA1_NCBI_mRNA.fasta"),
)

Fetching NP_000537.3 from NCBI db='protein' ...
Saved 465 characters to blast_results\TP53_NCBI_protein.fasta
Fetching NM_007294.4 from NCBI db='nucleotide' ...
Saved 7,281 characters to blast_results\BRCA1_NCBI_mRNA.fasta


In [6]:
# download function for Uniprot
UNIPROT_BASE = "https://rest.uniprot.org"
def download_from_uniprot(accession, file_format="fasta", filename=None):
    
    url = "%s/uniprotkb/%s.%s" % (UNIPROT_BASE, accession, file_format)
    print("GET", url)

    response = requests.get(url, timeout=60)
    response.raise_for_status()          # raise if 4xx / 5xx

    if filename is None:
        filename = os.path.join(OUTPUT_DIR, accession.replace(".", "_") + "." + file_format)

    with open(filename, "w") as f:
        f.write(response.text)

    print("Saved %s characters to %s" % (format(len(response.text), ","), filename))
    return filename


In [7]:
# download test protein from Uniprot
uniprot_file = download_from_uniprot(
    accession="P04637",                                    
    filename=os.path.join(OUTPUT_DIR, "TP53_UniProt.fasta"),
)

GET https://rest.uniprot.org/uniprotkb/P04637.fasta
Saved 490 characters to blast_results\TP53_UniProt.fasta


In [8]:
# quality control 
def gc_content(seq):
    """GC calculator"""
    seq = seq.upper()
    if not seq:
        return 0.0
    return 100.0 * (seq.count("G") + seq.count("C")) / len(seq)


def sequence_stats(path):
  
    def protein_mw_kDa(seq):

        try:
            from Bio.SeqUtils import molecular_weight
            try:
                mw = molecular_weight(seq, seq_type="protein")      
            except TypeError:
                mw = molecular_weight(seq, seq_format="protein")  
            return round(mw / 1000, 1)
        except Exception:
            return None

    rows = []
    for record in SeqIO.parse(path, "fasta"):
        seq = str(record.seq).upper()
        is_protein = bool(set(seq) - set("ACGTUN-"))
        row = {"file": os.path.basename(path), "id": record.id,
               "length": len(seq), "type": "protein" if is_protein else "nucleotide"}
        if is_protein:
            row["MW_kDa"] = protein_mw_kDa(record.seq)
        else:
            row["GC_%"] = round(gc_content(seq), 1)
        rows.append(row)
    return pd.DataFrame(rows)


In [9]:
pd.concat(
    [sequence_stats(f) for f in (ncbi_protein_file, ncbi_gene_file, uniprot_file)],
    ignore_index=True,
)

,file,id,length,type,MW_kDa,GC_%
0,TP53_NCBI_protein.fasta,NP_000537.3,393,protein,43.7,NaN
1,BRCA1_NCBI_mRNA.fasta,NM_007294.4,7088,nucleotide,NaN,41.8
2,TP53_UniProt.fasta,sp|P04637|P53_HUMAN,393,protein,43.7,NaN


In [10]:
def make_demo_fastq(path):
    """Write a tiny synthetic FASTQ file (2 reads) to demonstrate the format."""
    reads = [
        # (name, sequence, per-base Phred quality scores)
        ("read1", "ATGCGTACGTTAGCCAGGTAA", [40] * 21),                 # superb quality
        ("read2", "GGCCTTAAGGCATTCGATCGA", [30, 35, 38] + [25] * 18),  # tail degrading
    ]
    records = []
    for name, seq, quality in reads:
        rec = SeqRecord(Seq(seq), id=name, description="synthetic demo read")
        rec.letter_annotations["phred_quality"] = quality   # attach Q scores
        records.append(rec)
    SeqIO.write(records, path, "fastq")                     # SeqIO does the 4-line formatting
    return path


fastq_file = make_demo_fastq(os.path.join(OUTPUT_DIR, "demo.fastq"))

for record in SeqIO.parse(fastq_file, "fastq"):
    q = record.letter_annotations["phred_quality"]
    mean_q = sum(q) / len(q)
    # P(error) = 10 ** (-Q/10)  →  convert mean Q back to an accuracy estimate
    accuracy = 100 * (1 - 10 ** (-mean_q / 10))

In [11]:
# BLAST analysis function 
def run_blast(sequence, program="blastp", database="swissprot",
              expect=1e-3, hits=50, xml_file=None):

    if xml_file is None:
        xml_file = os.path.join(OUTPUT_DIR, "blast_%s_%s.xml" % (program, database))

    print("Submitting %s against '%s' (E <= %g, max %d hits) ..." % (program.upper(), database, expect, hits))
    t0 = time.time()

    result_handle = NCBIWWW.qblast(
        program=program,
        database=database,
        sequence=sequence,
        expect=expect,
        hitlist_size=hits,
    )
    xml_data = result_handle.read()
    result_handle.close()

    with open(xml_file, "w") as f:
        f.write(xml_data)

    print("Done in %.0f s -> %s (%s chars of XML)" % (time.time() - t0, xml_file, format(len(xml_data), ",")))
    return xml_file


In [ ]:
p53_record = SeqIO.read(uniprot_file, "fasta")

xml_file = run_blast(
    sequence=str(p53_record.seq),
    program="blastp",        
    database="swissprot",   
    expect=1e-5,            
    hits=25,
)


Submitting BLASTP against 'swissprot' (E <= 1e-05, max 25 hits) ...
